Problem 4: Naive Bayes classifier

In this problem you will implement your own Naive Bayes classifier and you will compare it with a package implementation. You will use the
Mushroom dataset for this problem. Split the dataset into 75% for training and 25% for testing.

In [27]:
import pandas as pd
import numpy as np
from sklearn.calibration import LabelEncoder
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
from ucimlrepo import fetch_ucirepo 
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score

In [28]:
# fetch dataset 
mushroom = fetch_ucirepo(id=73) 
  
# data (as pandas dataframes) 
X = mushroom.data.features 
y = mushroom.data.targets 
  
# metadata 
#print(mushroom.metadata) 
  
# variable information 
#print(mushroom.variables) 

In [29]:
# split data into training and test sets:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, shuffle=True)

1. Train the Naive Bayes classifier. Compute the prior probabilities for the Edible and Poisonous classes from the training data. For each feature $X_i$ in the dataset compute the probabilities $P[X_i = x| Y=\textit{Edible}]$, and $P[X_i = x| Y=\textit{Poisonous}]$ from the training data. Use the Laplace smoothing method when computing these probabilities. Note that the Naive Bayes classifier stores these prior and conditional probabilities.

In [30]:
# compute probabilities:
# probability of edible given feature and prob of poisonous given feature

# categorical naive bayes

y_train_poisonous = y_train['poisonous']
X_train_pd = pd.DataFrame(X_train).reset_index(drop=True)
y_train_pd = pd.Series(y_train_poisonous).reset_index(drop=True)

classes = y_train_pd.unique() # 2 classes: e or p
priors = {}
likelihoods = {}
N = len(y_train_pd) # total number of instances in traning set

for c in classes:
    X_c = X_train_pd[y_train_pd == c] # number of instances of this class
    n_c = len(X_c) 

    priors[c] = n_c / N # initial probability fo a class

    likelihoods[c] = {} # likelihood of feature occurring given class
    
    # final probability of a class given observed features
    for feature in X_train_pd.columns:
        k = X_train_pd[feature].nunique()
        likelihoods[c][feature] = {}
        for value in X_train_pd[feature].unique():
            count = (X_c[feature] == value).sum()
            # laplace smoothing: alpha = smoothing parameter
            alpha = 1
            likelihoods[c][feature][value] = (count + alpha) / (n_c + alpha * k)

likelihoods.keys()


dict_keys(['p', 'e'])

2. For each point in the testing set estimate the probability that it belongs to the Edible and Poisonous classes. Use the Naive Bayes
classifier probabilities computed in part (1).

In [35]:
def predict_proba_single(x):

    log_posteriors = {}
    classes = ['e', 'p']

    for c in classes:
        # ** use log for comparison **
        # start with log prior: log P(Y=c)
        log_prob = np.log(priors[c])

        # log likelihood for each feature: log P(Xi=x | Y=c)
        for feature, value in x.items():
            if value in likelihoods[c][feature]:
                log_prob += np.log(likelihoods[c][feature][value])
            else:
                # unseen value: Laplace smooth with count=0
                k = len(likelihoods[c][feature])
                log_prob += np.log(alpha / (sum(likelihoods[c][feature].values()) + alpha * k))

        log_posteriors[c] = log_prob        

    # convert log posteriors back to probabilities 
    max_log = max(log_posteriors.values())
    exp_vals = {}
    for c, lp in log_posteriors.items():
        exp_vals[c] = np.exp(lp - max_log)

    total = sum(exp_vals.values())
    result = {}
    for c in exp_vals.keys():
        result[c] = exp_vals[c] / total
    return result

def predict_proba(X):
    X = pd.DataFrame(X).reset_index(drop=True)
    probas = [predict_proba_single(row) for i, row in X.iterrows()]
    return pd.DataFrame(probas)

proba_df = predict_proba(X_test)
proba_df.columns = ['P(Edible)' if c == 'e' else 'P(Poisonous)' for c in proba_df.columns]

print("Predicted Probabilities for Test Set")
display(proba_df.head(10))

Predicted Probabilities for Test Set


,P(Edible),P(Poisonous)
0,9.999928e-01,7.186016e-06
1,4.514960e-10,1.000000e+00
2,2.164899e-11,1.000000e+00
3,1.000000e+00,1.687749e-08
4,4.310664e-10,1.000000e+00
5,8.290231e-11,1.000000e+00
6,1.675225e-03,9.983248e-01
7,9.026516e-17,1.000000e+00
8,1.000000e+00,8.909931e-09
9,9.999985e-01,1.483851e-06


3. Compute accuracy, precision, recall, and F1 score for your Naive Bayes classifier on the testing data.

In [32]:
def predict(X):
    proba_df = predict_proba(X)
    return proba_df.idxmax(axis=1)  

y_pred = predict(X_test)

print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred, pos_label='p'))
print("F1: ", f1_score(y_test, y_pred, pos_label='p'))

Accuracy:  0.9487936976858691
Recall:  0.9041372351160444
F1:  0.9451476793248945


4. Compare the results obtained by your implementation with those obtained with a Naive Bayes package (trained on the same dataset).
Use several metrics, including accuracy, precision, recall, and F1 score. Are the results similar or different?

In [33]:
# replace values that are NaN
X_train_clean = X_train.replace('?', np.nan)
X_test_clean  = X_test.replace('?', np.nan)

for col in X_train_clean.columns:
    mode = X_train_clean[col].mode()[0]
    X_train_clean[col] = X_train_clean[col].fillna(mode)
    X_test_clean[col]  = X_test_clean[col].fillna(mode)

# encode features and labels
encoder = OrdinalEncoder()
X_train_encoded = encoder.fit_transform(X_train_clean)
X_test_encoded  = encoder.transform(X_test_clean)

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded  = le.transform(y_test)

model = CategoricalNB(alpha=1)
model.fit(X_train_encoded, y_train_encoded)

/Users/pavithra/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_label.py:114: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/pavithra/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_label.py:132: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


CategoricalNB(alpha=1)

In [34]:
y_pred = model.predict(X_test_encoded)

print("Accuracy: ", accuracy_score(y_test_encoded, y_pred))
print("Recall: ", recall_score(y_test_encoded, y_pred))
print("F1: ", f1_score(y_test_encoded, y_pred))

Accuracy:  0.9502708025603152
Recall:  0.9091826437941474
F1:  0.9469259064634787


The results are approximately similar accross the board for accuracy, recall and F1 scores.